# 🏨 Hotel Revenue Portfolio Analysis
### A Portfolio Revenue Analyst's Perspective
**Andita Rachmania** | Data & Analytics

---

## Context & Objective

This notebook simulates the analytical work of a **Portfolio Revenue Analyst** — a role responsible for monitoring and optimising revenue performance across a portfolio of hotel properties.

The dataset used is the **Hotel Booking Demand** dataset (Antonio et al., 2019), containing ~119,000 real booking records from two properties: a **Resort Hotel** and a **City Hotel** spanning 2015–2017.

### What a Portfolio Revenue Analyst does with this data:
- Tracks key revenue KPIs: **RevPAR, ADR, Occupancy Rate**
- Monitors **booking pace** and demand patterns
- Analyses **market segmentation** and channel mix
- Flags **cancellation risk** and revenue leakage
- Produces insights to guide **pricing and commercial strategy**

### Sections
1. Environment Setup & Data Loading
2. Data Quality & Cleaning
3. KPI Engineering (RevPAR, ADR, Occupancy)
4. Temporal Analysis — Revenue Trends & Seasonality
5. Portfolio Comparison — Resort vs. City Hotel
6. Market Segmentation & Channel Mix
7. Booking Behaviour — Lead Time, Cancellations & Booking Pace
8. Geographic Analysis — Guest Origin
9. Revenue Opportunity Analysis
10. Key Insights & Recommendations

---
> 📌 **Dataset source**: [Kaggle – Hotel Booking Demand](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand)  
> Original paper: Antonio, N., de Almeida, A., & Nunes, L. (2019). Hotel booking demand datasets. *Data in Brief*.

---
## 1. Environment Setup & Data Loading

In [ ]:
# Install dependencies (run this if on Google Colab)
import subprocess, sys

packages = ['pandas', 'numpy', 'matplotlib', 'seaborn']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ All packages ready.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ── Plot styling ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f9f9f9',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

PALETTE = {'Resort Hotel': '#2E86AB', 'City Hotel': '#E84855'}

print('✅ Libraries loaded.')

In [ ]:
# ── Load dataset directly from public GitHub mirror (no Kaggle login needed) ──
DATA_URL = (
    'https://raw.githubusercontent.com/rfordatascience/tidytuesday/'
    'master/data/2020/2020-02-11/hotels.csv'
)

df_raw = pd.read_csv(DATA_URL)
print(f'✅ Dataset loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head(3)

---
## 2. Data Quality & Cleaning

> Before any KPI calculation, a revenue analyst must validate the data. Inaccurate inputs produce misleading insights — which can result in wrong pricing decisions.

In [ ]:
# ── Schema overview ────────────────────────────────────────────────────────────
print('SHAPE:', df_raw.shape)
print('\nCOLUMNS & DTYPES:')
print(df_raw.dtypes)
print('\nNULL COUNTS:')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

In [ ]:
# ── Key field summaries ────────────────────────────────────────────────────────
print('Hotel types:', df_raw['hotel'].value_counts().to_dict())
print('Years covered:', sorted(df_raw['arrival_date_year'].unique()))
print('Cancellation rate: {:.1%}'.format(df_raw['is_canceled'].mean()))
print('ADR range: ${:.0f} – ${:.0f}'.format(df_raw['adr'].min(), df_raw['adr'].max()))
print('Rows with ADR == 0:', (df_raw['adr'] == 0).sum())
print('Rows with no guest (adults+children+babies==0):',
      ((df_raw['adults'] + df_raw['children'].fillna(0) + df_raw['babies']) == 0).sum())

In [ ]:
# ── Cleaning pipeline ──────────────────────────────────────────────────────────
df = df_raw.copy()

# 1. Fill missing children values
df['children'] = df['children'].fillna(0)

# 2. Remove zero-ADR rows for stayed bookings (revenue can't be 0 for a real stay)
df = df[~((df['is_canceled'] == 0) & (df['adr'] <= 0))]

# 3. Remove bookings with no guests
df = df[(df['adults'] + df['children'] + df['babies']) > 0]

# 4. Remove extreme ADR outliers (above 99.5th percentile)
adr_cap = df['adr'].quantile(0.995)
df = df[df['adr'] <= adr_cap]

# 5. Build full arrival_date column
month_map = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
             'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}
df['arrival_month_num'] = df['arrival_date_month'].map(month_map)
df['arrival_date'] = pd.to_datetime(
    df['arrival_date_year'].astype(str) + '-' +
    df['arrival_month_num'].astype(str).str.zfill(2) + '-' +
    df['arrival_date_day_of_month'].astype(str).str.zfill(2),
    errors='coerce'
)
df['year_month'] = df['arrival_date'].dt.to_period('M')

# 6. Total stay length
df['total_stays'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']

# 7. Total revenue proxy (ADR × nights stayed)
df['room_revenue'] = df['adr'] * df['total_stays']

print(f'✅ Clean dataset: {df.shape[0]:,} rows (removed {df_raw.shape[0] - df.shape[0]:,} anomalies)')

---
## 3. KPI Engineering: RevPAR, ADR, Occupancy

The three pillars of hotel revenue performance:

| KPI | Formula | What it tells us |
|---|---|---|
| **ADR** | Total Room Revenue / Rooms Sold | How much guests pay on average per night |
| **Occupancy Rate** | Rooms Sold / Rooms Available | What share of inventory is being filled |
| **RevPAR** | ADR × Occupancy Rate | Revenue generated per *available* room — the master KPI |

> Since we don't have total room inventory, we **approximate Occupancy** as the share of bookings that were *stayed* (not cancelled) relative to total bookings received. This is a normalised relative measure.

In [ ]:
# ── Portfolio-level KPI summary ────────────────────────────────────────────────
def portfolio_kpis(data, label='Portfolio'):
    stayed = data[data['is_canceled'] == 0]
    total  = len(data)
    stayed_n = len(stayed)

    adr         = stayed['adr'].mean()
    occ_rate    = stayed_n / total if total > 0 else 0
    revpar      = adr * occ_rate
    cancel_rate = data['is_canceled'].mean()
    avg_lead    = data['lead_time'].mean()
    avg_stay    = stayed['total_stays'].mean()
    total_rev   = stayed['room_revenue'].sum()

    return {
        'Segment'             : label,
        'Total Bookings'      : total,
        'Stayed Bookings'     : stayed_n,
        'Cancellation Rate'   : f'{cancel_rate:.1%}',
        'ADR ($)'             : f'{adr:.2f}',
        'Occupancy (rel.)'    : f'{occ_rate:.1%}',
        'RevPAR ($)'          : f'{revpar:.2f}',
        'Avg Lead Time (days)': f'{avg_lead:.0f}',
        'Avg Stay (nights)'   : f'{avg_stay:.1f}',
        'Est. Total Revenue ($)': f'{total_rev:,.0f}'
    }

# Overall + per property
kpi_rows = [
    portfolio_kpis(df, 'All Properties'),
    portfolio_kpis(df[df['hotel']=='Resort Hotel'], 'Resort Hotel'),
    portfolio_kpis(df[df['hotel']=='City Hotel'],   'City Hotel'),
]

kpi_df = pd.DataFrame(kpi_rows).set_index('Segment')
print('=== PORTFOLIO KPI SUMMARY ===')
kpi_df.T

---
## 4. Temporal Analysis — Revenue Trends & Seasonality

> A core deliverable for any portfolio analyst: **how are our properties performing month over month?**  
> We track RevPAR trends to identify seasonality, growth, and underperforming periods.

In [ ]:
# ── Monthly KPIs per property ──────────────────────────────────────────────────
stayed_df = df[df['is_canceled'] == 0].copy()
all_df    = df.copy()

monthly_stayed = (
    stayed_df.groupby(['year_month', 'hotel'])
    .agg(stayed_bookings=('adr', 'count'), adr=('adr', 'mean'), revenue=('room_revenue', 'sum'))
    .reset_index()
)

monthly_total = (
    all_df.groupby(['year_month', 'hotel'])
    .agg(total_bookings=('adr', 'count'))
    .reset_index()
)

monthly = monthly_stayed.merge(monthly_total, on=['year_month','hotel'])
monthly['occupancy'] = monthly['stayed_bookings'] / monthly['total_bookings']
monthly['revpar']    = monthly['adr'] * monthly['occupancy']
monthly['period']    = monthly['year_month'].astype(str)

monthly.head(4)

In [ ]:
# ── Plot: Monthly RevPAR trend ─────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 13), sharex=True)

metrics = [
    ('revpar',    'RevPAR ($)',    'Revenue Per Available Room'),
    ('adr',       'ADR ($)',       'Average Daily Rate'),
    ('occupancy', 'Occupancy',     'Relative Occupancy Rate'),
]

for ax, (col, ylabel, title) in zip(axes, metrics):
    for hotel, grp in monthly.groupby('hotel'):
        grp_sorted = grp.sort_values('period')
        ax.plot(grp_sorted['period'], grp_sorted[col],
                marker='o', ms=3, linewidth=2,
                label=hotel, color=PALETTE[hotel])

    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.legend(loc='upper left', fontsize=9)
    if col == 'occupancy':
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.tick_params(axis='x', labelrotation=45, labelsize=7)

plt.suptitle('Monthly Revenue Performance — Portfolio View (2015–2017)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Seasonality heatmap ────────────────────────────────────────────────────────
pivot = (
    monthly[monthly['hotel'] == 'Resort Hotel']
    .assign(year=lambda x: x['year_month'].dt.year,
            month=lambda x: x['year_month'].dt.month)
    .pivot_table(index='month', columns='year', values='revpar', aggfunc='mean')
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, hotel in zip(axes, ['Resort Hotel', 'City Hotel']):
    pivot = (
        monthly[monthly['hotel'] == hotel]
        .assign(year=lambda x: x['year_month'].dt.year,
                month=lambda x: x['year_month'].dt.month)
        .pivot_table(index='month', columns='year', values='revpar', aggfunc='mean')
    )
    pivot.index = ['Jan','Feb','Mar','Apr','May','Jun',
                   'Jul','Aug','Sep','Oct','Nov','Dec'][:len(pivot)]
    sns.heatmap(pivot, ax=ax, cmap='YlOrRd', annot=True, fmt='.0f',
                linewidths=0.5, cbar_kws={'label': 'RevPAR ($)'})
    ax.set_title(f'{hotel} — RevPAR Heatmap by Month & Year', fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Month')

plt.tight_layout()
plt.show()

---
## 5. Portfolio Comparison — Resort vs. City Hotel

> Portfolio analysts must understand each property's **strengths, weaknesses, and strategic positioning** within the portfolio. Not all hotels should be managed the same way.

In [ ]:
# ── ADR distribution by property ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ADR distributions
for hotel, grp in stayed_df.groupby('hotel'):
    axes[0].hist(grp['adr'], bins=60, alpha=0.65, label=hotel,
                 color=PALETTE[hotel], edgecolor='white')
axes[0].set_xlabel('ADR ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('ADR Distribution by Property', fontweight='bold')
axes[0].legend()

# Stay length distribution
for hotel, grp in stayed_df.groupby('hotel'):
    stay_counts = grp['total_stays'].value_counts().sort_index()
    stay_counts = stay_counts[stay_counts.index <= 14]
    axes[1].bar(stay_counts.index + (0.2 if hotel == 'City Hotel' else -0.2),
                stay_counts.values, width=0.4, alpha=0.85,
                label=hotel, color=PALETTE[hotel])
axes[1].set_xlabel('Length of Stay (nights)')
axes[1].set_ylabel('Bookings')
axes[1].set_title('Length of Stay Distribution', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── KPI spider / bar comparison ────────────────────────────────────────────────
compare_metrics = ['ADR ($)', 'RevPAR ($)', 'Cancellation Rate', 'Avg Lead Time (days)', 'Avg Stay (nights)']
compare_data = kpi_df.loc[['Resort Hotel','City Hotel'], compare_metrics].copy()

# Convert to numeric for plotting
numeric_cols = ['ADR ($)', 'RevPAR ($)', 'Avg Lead Time (days)', 'Avg Stay (nights)']
compare_num = compare_data.copy()
for col in numeric_cols:
    compare_num[col] = pd.to_numeric(compare_data[col], errors='coerce')
compare_num['Cancellation Rate'] = compare_data['Cancellation Rate'].str.rstrip('%').astype(float)

fig, axes = plt.subplots(1, len(numeric_cols), figsize=(16, 4))

for ax, col in zip(axes, numeric_cols):
    hotels = compare_num.index.tolist()
    vals   = compare_num[col].tolist()
    bars = ax.bar(hotels, vals, color=[PALETTE[h] for h in hotels], alpha=0.85, edgecolor='white')
    ax.set_title(col, fontweight='bold')
    ax.set_xticks(range(len(hotels)))
    ax.set_xticklabels(['Resort', 'City'], fontsize=10)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Property Comparison: Key Revenue Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Market Segmentation & Channel Mix

> Revenue strategy is not just about price — it's about **who books, through which channel, and at what rate**. A healthy mix reduces dependency on high-commission channels (like OTAs) and improves net revenue.

In [ ]:
# ── Market segment: volume + ADR + cancellation rate ──────────────────────────
seg_analysis = (
    df.groupby(['hotel', 'market_segment'])
    .agg(
        total_bookings   = ('is_canceled', 'count'),
        cancellation_rate= ('is_canceled', 'mean'),
        avg_adr          = ('adr', 'mean'),
    )
    .reset_index()
)

seg_analysis['share'] = (
    seg_analysis.groupby('hotel')['total_bookings']
    .transform(lambda x: x / x.sum())
)

print('Market Segment Summary:')
seg_analysis.sort_values(['hotel','total_bookings'], ascending=[True, False]).to_string(index=False)

In [ ]:
# ── Plot: Segment share + ADR by property ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for row, hotel in enumerate(['Resort Hotel', 'City Hotel']):
    hotel_seg = seg_analysis[seg_analysis['hotel'] == hotel].sort_values('total_bookings', ascending=False)

    # Booking share (pie)
    axes[row][0].pie(
        hotel_seg['share'],
        labels=hotel_seg['market_segment'],
        autopct='%1.1f%%',
        startangle=140,
        colors=sns.color_palette('Set2', len(hotel_seg))
    )
    axes[row][0].set_title(f'{hotel}\nBooking Share by Market Segment', fontweight='bold')

    # ADR + Cancellation by segment
    x = range(len(hotel_seg))
    ax2 = axes[row][1].twinx()
    bars = axes[row][1].bar(x, hotel_seg['avg_adr'], alpha=0.75,
                            color=PALETTE[hotel], label='Avg ADR')
    ax2.plot(x, hotel_seg['cancellation_rate'], 'o--', color='#E07A5F',
             linewidth=2, markersize=7, label='Cancel Rate')
    axes[row][1].set_xticks(list(x))
    axes[row][1].set_xticklabels(hotel_seg['market_segment'], rotation=30, ha='right', fontsize=8)
    axes[row][1].set_ylabel('Avg ADR ($)')
    ax2.set_ylabel('Cancellation Rate')
    ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    axes[row][1].set_title(f'{hotel}\nADR vs. Cancellation Rate by Segment', fontweight='bold')
    axes[row][1].legend(loc='upper left', fontsize=8)
    ax2.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

---
## 7. Booking Behaviour — Lead Time, Cancellations & Booking Pace

> **Booking pace** — how fast bookings accumulate for a future date — is one of the most operationally critical metrics for a revenue analyst. It signals whether to hold rates, open discounts, or restrict inventory.

> **Lead time** tells us when demand is committed. Long lead times = more planning visibility; short lead times = last-minute demand requiring dynamic pricing.

In [ ]:
# ── Lead time distribution ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram (clipped at 365 days for readability)
for hotel, grp in df.groupby('hotel'):
    lead_clipped = grp['lead_time'].clip(upper=365)
    axes[0].hist(lead_clipped, bins=50, alpha=0.6,
                 label=hotel, color=PALETTE[hotel], edgecolor='white')
axes[0].set_xlabel('Lead Time (days, capped at 365)')
axes[0].set_ylabel('Bookings')
axes[0].set_title('Booking Lead Time Distribution', fontweight='bold')
axes[0].legend()

# Cumulative lead time (booking pace simulation)
bins = [0, 7, 14, 30, 60, 90, 180, 365, 10000]
labels = ['0-7d', '8-14d', '15-30d', '31-60d', '61-90d', '91-180d', '181-365d', '365d+']
for hotel, grp in df.groupby('hotel'):
    bucketed = pd.cut(grp['lead_time'], bins=bins, labels=labels, right=True)
    bucket_pct = bucketed.value_counts(normalize=True).sort_index()
    axes[1].plot(labels, bucket_pct.values, marker='o', linewidth=2,
                 label=hotel, color=PALETTE[hotel])
axes[1].set_xlabel('Lead Time Window')
axes[1].set_ylabel('Share of Bookings')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].set_title('Booking Volume by Lead Time Window', fontweight='bold')
axes[1].tick_params(axis='x', labelrotation=30)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Cancellation analysis ──────────────────────────────────────────────────────
cancel_by_seg = (
    df.groupby(['hotel','market_segment'])['is_canceled']
    .agg(['mean','count'])
    .rename(columns={'mean':'cancel_rate','count':'n_bookings'})
    .reset_index()
    .sort_values(['hotel','cancel_rate'], ascending=[True,False])
)

# Cancellation rate by lead time bucket
df['lead_bucket'] = pd.cut(
    df['lead_time'],
    bins=[0, 7, 14, 30, 60, 90, 180, 365, 10000],
    labels=['0-7d','8-14d','15-30d','31-60d','61-90d','91-180d','181-365d','365d+']
)

cancel_by_lead = (
    df.groupby(['hotel','lead_bucket'])['is_canceled']
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Cancellation by segment
pivot_cancel = cancel_by_seg.pivot(index='market_segment', columns='hotel', values='cancel_rate')
pivot_cancel.plot(kind='barh', ax=axes[0], color=[PALETTE['Resort Hotel'], PALETTE['City Hotel']],
                  alpha=0.85, edgecolor='white')
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[0].set_title('Cancellation Rate by Market Segment', fontweight='bold')
axes[0].set_xlabel('Cancellation Rate')
axes[0].legend(title='Hotel')

# Cancellation by lead time
for hotel, grp in cancel_by_lead.groupby('hotel'):
    axes[1].plot(grp['lead_bucket'].astype(str), grp['is_canceled'],
                 marker='o', linewidth=2, label=hotel, color=PALETTE[hotel])
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].set_title('Cancellation Rate by Lead Time Window\n(Longer bookings = higher risk)', fontweight='bold')
axes[1].set_xlabel('Lead Time')
axes[1].tick_params(axis='x', labelrotation=35)
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 8. Geographic Analysis — Guest Origin

> Understanding **where guests come from** shapes pricing strategy, marketing spend, and room mix decisions. High-value source markets deserve targeted yield management.

In [ ]:
# ── Top 15 guest markets by stayed bookings ────────────────────────────────────
geo = (
    stayed_df.groupby(['hotel','country'])
    .agg(bookings=('adr','count'), avg_adr=('adr','mean'), avg_stay=('total_stays','mean'))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, hotel in zip(axes, ['Resort Hotel', 'City Hotel']):
    top = geo[geo['hotel']==hotel].nlargest(15, 'bookings').sort_values('bookings')
    bars = ax.barh(top['country'], top['bookings'], color=PALETTE[hotel], alpha=0.85, edgecolor='white')
    # Add ADR labels
    for bar, adr_val in zip(bars, top['avg_adr']):
        ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                f'ADR ${adr_val:.0f}', va='center', fontsize=7.5)
    ax.set_title(f'{hotel}\nTop 15 Guest Markets (stayed bookings)', fontweight='bold')
    ax.set_xlabel('Stayed Bookings')

plt.tight_layout()
plt.show()

---
## 9. Revenue Opportunity Analysis

> Identifying **where revenue is being left on the table** is the highest-value output a portfolio analyst can produce. We look at:
> - Revenue lost to cancellations
> - ADR gap between channels
> - High-volume, low-ADR segments (candidates for rate floor adjustments)

In [ ]:
# ── Revenue lost to cancellations ──────────────────────────────────────────────
cancelled_df = df[df['is_canceled'] == 1].copy()

# Estimate lost revenue: use the ADR of the cancelled booking × expected 2-night stay
# (conservative estimate — we don't know what stay length would have been)
cancelled_df['est_lost_revenue'] = cancelled_df['adr'] * 2  # 2-night proxy

lost_by_hotel = cancelled_df.groupby('hotel')['est_lost_revenue'].sum().reset_index()
lost_by_hotel.columns = ['Hotel', 'Est. Revenue Lost ($)']

lost_by_seg = (
    cancelled_df.groupby(['hotel','market_segment'])['est_lost_revenue']
    .sum()
    .reset_index()
    .sort_values('est_lost_revenue', ascending=False)
)

print('=== ESTIMATED REVENUE LOST TO CANCELLATIONS (2-night proxy) ===')
print(lost_by_hotel.to_string(index=False))
print('\nTop segments driving lost revenue:')
print(lost_by_seg.head(8).to_string(index=False))

In [ ]:
# ── ADR opportunity: volume vs. rate bubble chart ──────────────────────────────
bubble = (
    df[df['is_canceled']==0]
    .groupby(['hotel','market_segment'])
    .agg(n=('adr','count'), avg_adr=('adr','mean'), cancel_rate=('is_canceled','mean'))
    .reset_index()
)

# Re-join cancel rate from full df
cancel_ref = df.groupby(['hotel','market_segment'])['is_canceled'].mean().reset_index()
cancel_ref.columns = ['hotel','market_segment','cancel_rate']
bubble = bubble.drop(columns=['cancel_rate']).merge(cancel_ref, on=['hotel','market_segment'])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, hotel in zip(axes, ['Resort Hotel', 'City Hotel']):
    sub = bubble[bubble['hotel'] == hotel]
    sc = ax.scatter(
        sub['avg_adr'], sub['cancel_rate'],
        s=sub['n'] / 5,
        alpha=0.7, color=PALETTE[hotel], edgecolors='white', linewidths=0.5
    )
    for _, row in sub.iterrows():
        ax.annotate(row['market_segment'], (row['avg_adr'], row['cancel_rate']),
                    fontsize=8, ha='left', va='bottom',
                    xytext=(4, 4), textcoords='offset points')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_xlabel('Avg ADR ($)')
    ax.set_ylabel('Cancellation Rate')
    ax.set_title(f'{hotel}\nADR vs. Cancellation Rate\n(bubble size = booking volume)', fontweight='bold')
    ax.axhline(sub['cancel_rate'].mean(), linestyle='--', color='grey', alpha=0.6, label='Portfolio avg cancel rate')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print('\n💡 Segments in top-right quadrant (high ADR + high cancellation) = high-risk revenue — monitor closely.')
print('💡 Segments in bottom-left quadrant (low ADR + low cancellation) = stable but underpriced — consider rate floor strategy.')

---
## 10. Key Insights & Analyst Recommendations

> This section synthesises the analysis into actionable insights — the core output expected from a Portfolio Revenue Analyst.


In [ ]:
# ── Summary KPI card print ─────────────────────────────────────────────────────
print('=' * 60)
print('  PORTFOLIO REVENUE PERFORMANCE SUMMARY')
print('=' * 60)

for hotel in ['Resort Hotel', 'City Hotel', 'All Properties']:
    row = kpi_df.loc[hotel]
    print(f'\n📍 {hotel}')
    print(f'   ADR:               {row["ADR ($)"]}')
    print(f'   RevPAR:            {row["RevPAR ($)"]}')
    print(f'   Occupancy (rel.):  {row["Occupancy (rel.)"]}')
    print(f'   Cancellation Rate: {row["Cancellation Rate"]}')
    print(f'   Avg Lead Time:     {row["Avg Lead Time (days)"]} days')
    print(f'   Avg Stay Length:   {row["Avg Stay (nights)"]} nights')

### 📋 Analyst Recommendations

Based on the portfolio analysis above, here are the strategic takeaways:

---

**1. Cancellation Rate is the #1 Revenue Leak**  
Both properties show high cancellation rates (typically 25–45%). The data shows cancellations spike as lead time increases — bookings made 90+ days in advance are significantly more likely to cancel. **Recommendation**: introduce non-refundable rate options for far-out booking windows, or tighten deposit requirements for long lead-time reservations.

**2. Online TA is Dominant but Risky**  
Online Travel Agencies (OTAs) drive the highest booking volume but also show elevated cancellation rates. High OTA commission (typically 15–25%) further erodes net revenue. **Recommendation**: incentivise direct bookings through loyalty programmes and best-rate guarantees to improve channel mix and net RevPAR.

**3. City Hotel Needs Rate Floor Protection**  
The City Hotel shows high booking volume but often at lower ADR than the Resort — particularly in corporate and group segments. **Recommendation**: review rate floors for low-ADR segments and consider restricting availability during high-demand periods.

**4. Seasonal Peaks Are Predictable — Use Them for Yield Management**  
The heatmap clearly shows summer peaks (July–August) for the Resort and a more stable year-round pattern for the City Hotel. **Recommendation**: proactively increase rates 90–120 days before peak periods, and use restrictive stay patterns (minimum stay requirements) to optimise mix during compression periods.

**5. Market Segmentation Drives Strategy**  
Each segment has a distinct ADR and cancellation profile. **Recommendation**: build a segment-level performance scorecard reviewed monthly at revenue meetings to track performance drift early.

---
> *Analysis by Andita Rachmania | Dataset: Hotel Booking Demand (Antonio et al., 2019)*